In [14]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [15]:
#Load the dataset
df=pd.read_csv("netflix_titles.csv")

In [16]:
#....Data cleaning and preprocessing...

#Fill now in relevant columns with an empty string
features=['director','cast','listed_in','description']
for feature in features:
    df[feature]=df[feature].fillna("")
    

In [17]:
#Function to get up to 3 names/items from a comma-separated string
#and simplify them by lowercasing and removing spaces(e.g.,'Tom Hanks'--> 'tamhanks)
def get_list_limit_and_clean(obj,limit=3):
    L=obj.split(',')
    if len(L)>limit:
        L=L[limit]
    #simplify names:lowecase and remove spaces
    return[x.lower().replace(' ','')for x in L]

In [18]:
#Apply the cleaning/limiting to direcrtor,cast and genres
#Director is kept as a list for easy concatenation later

df['director']=df['director'].apply(lambda x:[x.lower().replace(' ','')])
df['cast']=df['cast'].apply(get_list_limit_and_clean)
df['listed_in']=df['listed_in'].apply(get_list_limit_and_clean)
df['description']=df['description'].apply(get_list_limit_and_clean)

In [22]:
#--2.create the weighted Metadata soup--

def create_soup(x):
    '''
    combine all features into a single string(the 'soup').
    Director/cast/genres are repeated 3 times for weighting.
    '''
    '''#metadata list=[dircetor_list]+[cast_list]+[genre_list]
    metadata_list=x['director']+x['cast']+x['listed_in']

    #Repeat the high-value metadata 3 times for weighting,then add the description
    return' '.join(metadata_list*3)+' '+x['description']
df['soup']=df.apply(create_soup,axis=1)
def create_soup(x):'''
    
    # Ensure metadata fields are lists
    director = x['director'] if isinstance(x['director'], list) else []
    cast = x['cast'] if isinstance(x['cast'], list) else []
    listed_in = x['listed_in'] if isinstance(x['listed_in'], list) else []
    
    # Ensure description is string
    description = x['description']
    if isinstance(description, list):
        description = ' '.join(description)
    elif not isinstance(description, str):
        description = ''
    
    metadata_list = director + cast + listed_in
    
    return ' '.join(metadata_list * 3) + ' ' + description


df['soup'] = df.apply(create_soup, axis=1)


In [23]:
df['soup'].head()

0    kirstenjohnson  documentaries kirstenjohnson  ...
1      t h a b a n g  m o l a b a internationaltvsh...
2    julienleclercq  n a b i h a  a k k a r i crime...
3      docuseries realitytv   docuseries realitytv ...
4      a l a m  k h a n internationaltvshows romant...
Name: soup, dtype: object

In [24]:
#content vectorization --(TF-IDF)---
#Initialize the TF-IDF vectorizer
#max features limits the vocabulary  size

tfidf=TfidfVectorizer(stop_words='english',max_features=20000)

#fit and transform the 'soup'
tfidf_matrix=tfidf.fit_transform(df['soup'])


In [26]:
#calculate cosine similarity 
cosine_sim=cosine_similarity(tfidf_matrix,tfidf_matrix)

In [28]:
print(cosine_sim)

[[1.         0.         0.         ... 0.         0.         0.        ]
 [0.         1.         0.07474075 ... 0.         0.         0.        ]
 [0.         0.07474075 1.         ... 0.         0.         0.        ]
 ...
 [0.         0.         0.         ... 1.         0.06251298 0.        ]
 [0.         0.         0.         ... 0.06251298 1.         0.        ]
 [0.         0.         0.         ... 0.         0.         1.        ]]


In [33]:
#Enhanced recommender function
#reset index and create a reverse map for the quick title lookup

df=df.reset_index()
indices=pd.Series(df.index,index=df['title']).drop_duplicates()

def get_enhanced_recommendations(title,cosine_sim_matrix=cosine_sim,df_data=df,indices_map=indices,top_n=10):
    '''
    Generates enhanced content based recommendation for a given item title.
    '''
    try:
        #get the index of the item that matches the title
        idx=indices_map[title]
    except KeyError:
        return f"Error:'{title}' not found in the dataset.Please check the spelling."
    #get the similarity scores for that item with all other items
    
    sim_scores=list(enumerate(cosine_sim_matrix[idx]))

    #sort items based on similarity scores-descending order
    sim_scores=sorted(sim_scores,key=lambda x:x[1],reverse=True)

    #4.get the scores of the top_n mostmsimilar items (excluding itself)
    #the first score(index 0) will be the itself (score=1.0)
    sim_scores=sim_scores[1:top_n+1]

    #5.get the item indices
    item_indices=[i[0] for i in sim_scores]

    #6.return the titles of the most similar items
    recommendations=df_data['title'].iloc[item_indices].tolist()

    return recommendations

In [35]:
#---6.Demonstration and testing---
#test Case 1:Tv show

test_title_1='Blood & Water'
recs_1=get_enhanced_recommendations(test_title_1)
print(f"Recommendations for '{test_title_1}' (TV show):")
for i,rec in enumerate(recs_1,1):
    print(f"{i}.{rec}")
print("-"*30)



Recommendations for 'Blood & Water' (TV show):
1.The Gift
2.We Are the Wave
3.Jinn
4.Into the Night
5.Open Your Eyes
6.More to Say
7.Switched
8.Girl from Nowhere
9.Between
10.Katla
------------------------------


In [37]:
#test Case 1:movie (documentary)

test_title_2='Dick Johnson Is Dead'
recs_2=get_enhanced_recommendations(test_title_2)
print(f"Recommendations for '{test_title_2}' (Documentary):")
for i,rec in enumerate(recs_2,1):
    print(f"{i}.{rec}")
print("-"*30)

Recommendations for 'Dick Johnson Is Dead' (Documentary):
1.Crip Camp: A Disability Revolution
2.A new Capitalism
3.Headspace: Unwind Your Mind
4.Free to Play
5.The Minimalists: Less Is Now
6.The Irishman: In Conversation
7.American Factory: A Conversation with the Obamas
8.Saudi Arabia Uncovered
9.Real Crime: Supermarket Heist (Tesco Bomber)
10.She Did That
------------------------------
